In [59]:
#Initial Imports
from astropy.io import fits
from scipy.ndimage import median_filter
import numpy as np
from photutils.detection import DAOStarFinder
from scipy.optimize import curve_fit, linear_sum_assignment
import matplotlib.pyplot as plt
import math
import pandas as pd
import re
import matplotlib.colors as mcolors
import colorsys
import os


In [61]:
#list of fits file paths paired with detector temperatures at the time of observation
#stored as a list of tuples (filepath, temp(float))
temp_pairs = [(r"Y:\2D\20260406\iLocater_lab_20260406_0001_hxrgproc.fits", 130.738),
(r"Y:\2D\20260406\iLocater_lab_20260406_0002_hxrgproc.fits", 130.755),
(r"Y:\2D\20260407\iLocater_lab_20260407_0001_hxrgproc.fits", 121.429),
(r"Y:\2D\20260407\iLocater_lab_20260407_0002_hxrgproc.fits", 121.461),
(r"Y:\2D\20260409\iLocater_lab_20260409_0001_hxrgproc.fits", 102.087),
(r"Y:\2D\20260409\iLocater_lab_20260409_0002_hxrgproc.fits", 102.078),
(r"Y:\2D\20260410\iLocater_lab_20260410_0001_hxrgproc.fits", 95.8237),
(r"Y:\2D\20260410\iLocater_lab_20260410_0002_hxrgproc.fits", 95.8189),
(r"Y:\2D\20260411\iLocater_lab_20260411_0001_hxrgproc.fits", 90.8267),
(r"Y:\2D\20260411\iLocater_lab_20260411_0002_hxrgproc.fits", 90.8215),
(r"Y:\2D\20260412\iLocater_lab_20260412_0001_hxrgproc.fits", 88.1519),
(r"Y:\2D\20260412\iLocater_lab_20260412_0002_hxrgproc.fits", 88.1490),
(r"Y:\2D\20260413\iLocater_lab_20260413_0001_hxrgproc.fits", 86.4587),
(r"Y:\2D\20260413\iLocater_lab_20260413_0002_hxrgproc.fits", 86.4556),
(r"Y:\2D\20260414\iLocater_lab_20260414_0001_hxrgproc.fits", 84.8636),
(r"Y:\2D\20260414\iLocater_lab_20260414_0002_hxrgproc.fits", 84.8621),
(r"Y:\2D\20260415\iLocater_lab_20260415_0001_hxrgproc.fits", 83.7054),
(r"Y:\2D\20260415\iLocater_lab_20260415_0002_hxrgproc.fits", 83.7017),
(r"Y:\2D\20260416\iLocater_lab_20260416_0016_hxrgproc.fits", 84.1627),
(r"Y:\2D\20260416\iLocater_lab_20260416_0017_hxrgproc.fits", 84.1642),
(r"Y:\2D\20260417\iLocater_lab_20260417_0001_hxrgproc.fits", 84.41245),
(r"Y:\2D\20260417\iLocater_lab_20260417_0002_hxrgproc.fits", 84.4123),
(r"Y:\2D\20260420\iLocater_lab_20260420_0003_hxrgproc.fits", 84.69485),
(r"Y:\2D\20260420\iLocater_lab_20260420_0004_hxrgproc.fits", 84.6964),
(r"Y:\2D\20260421\iLocater_lab_20260421_0012_hxrgproc.fits", 84.69515),
(r"Y:\2D\20260421\iLocater_lab_20260421_0013_hxrgproc.fits", 84.6944),
(r"Y:\2D\20260422\iLocater_lab_20260422_0001_hxrgproc.fits", 84.6951),
(r"Y:\2D\20260422\iLocater_lab_20260422_0002_hxrgproc.fits", 84.69465),
(r"Y:\2D\20260423\iLocater_lab_20260423_0016_hxrgproc.fits", 84.6940),
(r"Y:\2D\20260423\iLocater_lab_20260423_0017_hxrgproc.fits", 84.6929),
(r"Y:\2D\20260424\iLocater_lab_20260424_0001_hxrgproc.fits", 84.6954),
(r"Y:\2D\20260424\iLocater_lab_20260424_0002_hxrgproc.fits", 84.6454)]


In [62]:
"""the name of whatever folder you wish to contain the plots generated 
by the notebook. Set as "None" if you want to display in the notebook 
instead of saving them.
 """
plot_output_path = "per_amp_temp_plots"
os.makedirs(plot_output_path, exist_ok=True)

In [63]:
def process_amps(temp_pairs):
    rows = []
    for path, temp in temp_pairs:
        data = fits.getdata(path)
        sections = np.split(data, 32, axis=1)
        
        #calculate summary stats for each amplifier
        for i, amp in enumerate(sections):
            n = np.size(amp)
            total = np.sum(amp)
            mean = np.mean(amp)
            std = np.std(amp)
            sem = std / np.sqrt(n)
            var = std**2

            rows.append({
                "Filename": os.path.basename(path),
                "Amp #": i + 1,
                "Temp": temp,
                "Total Readout": total,
                "Total Readout Uncertainty": n * sem,
                "Mean Readout (Per Pixel)": mean,
                "Standard Deviation (Per Pixel)": std,
                "Standard Error in the Pixel Mean": sem,
                "Variance (Per Pixel)": var
            })
    return pd.DataFrame(rows)

In [64]:
df = process_amps(temp_pairs)

In [65]:
even = np.arange(len(temp_pairs)) % 2 == 0
odd = ~even
def plot_amp_vs_temp(amp_df, amp, even=even, odd=odd, savedir=None):
    t = amp_df["Temp"].to_numpy()
    y1 = amp_df["Total Readout"].to_numpy()
    y1err = amp_df["Total Readout Uncertainty"].to_numpy()
    y2 = amp_df["Mean Readout (Per Pixel)"].to_numpy()
    y2err = amp_df["Standard Error in the Pixel Mean"].to_numpy()

    fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7,8))
    fig.suptitle(f"Stats By Temp for Amp #{amp}")

    #plot total readout data for short exposures (even) and long exposures (odd)
    axes[0].errorbar(t[even], y1[even], y1err[even], 
    c="darkgoldenrod", fmt="o", capsize=4, markersize=3, alpha=0.75, label="Short Exposure")
    axes[0].errorbar(t[odd], y1[odd], y1err[odd],
    c="darkslategrey", fmt="o", capsize=4, markersize=3, alpha=0.75, label="Long Exposure")
    axes[0].set_ylabel("Amp-Wide Dark Current (ADU/Frame)")
    axes[0].legend()

    #plot mean pixel readout, even and odd distinction is the same
    axes[1].errorbar(t[even], y2[even], y2err[even],
    c="mediumorchid", fmt="o", capsize=4, markersize=3, alpha=0.75, label="Short Exposure")
    axes[1].errorbar(t[odd], y2[odd], y2err[odd],
    c="olive", fmt="o", capsize=4, markersize=3, alpha=0.75, label="Long Exposure")
    axes[1].set_ylabel("Mean Pixel Dark Current (ADU/Pixel/Frame)")
    axes[1].set_xlabel("Detector Temp (K)")
    axes[1].legend()
    plt.tight_layout()

    #save plot to the supplied folder directory if one is supplied
    if savedir is not None:
        out_path = os.path.join(savedir, f"amp_{amp:02d}.png")
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()
    else:
        #display subplots if we aren't saving them
        plt.show()

In [66]:
#generate plots for all 32 amps
for amp in range (1,33):
    plot_amp_vs_temp(df[df["Amp #"] == amp], amp, even, odd, savedir = plot_output_path)